# 1. Import Libraries

In [1]:
import os 
import rasterio 
import numpy as np 
import albumentations as A

c:\Users\Asmaa\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Dataset Configuration

In [2]:
Data_path = r"D:\master\02- Second Semester\04- AI\Project\EuroSAT_MS\EuroSAT_MS"

final_dataset_path = os.path.join(
    os.path.dirname(Data_path),
    "Processed_Data"
)

# 3. Image Processing Functions


In [3]:
def process_tiff_image(image_path, bands=range(1, 14)):
    with rasterio.open(image_path) as src:        
        img = src.read(list(bands)).astype(np.float32)        
        img = np.transpose(img, (1, 2, 0))       
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

In [4]:
def process_rgb_image(image_path, bands=[4, 3, 2]):   
    with rasterio.open(image_path) as src:        
        img = src.read(bands).astype(np.float32)        
        img = np.transpose(img, (1, 2, 0))        
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

In [5]:
# Data Augmentation
augmentation = A.Compose([   
    A.HorizontalFlip(p=0.5),    
    A.VerticalFlip(p=0.5),    
    A.RandomBrightnessContrast(p=0.2)
])


# 4. Feature Engineering


In [6]:
def compute_ndvi(image, nir_band=3, red_band=0):    
    nir = image[:, :, nir_band].astype(float)   
    red = image[:, :, red_band].astype(float)    
    ndvi = (nir - red) / (nir + red + 1e-5)
    return ndvi

In [7]:
def create_rgb_nir(image_path):    
    rgb_nir = process_tiff_image(
        image_path,
        bands=[4, 3, 2, 8])   
    return rgb_nir

# 5. Dataset Export

In [8]:
def save_all_processed_images(data_path, final_dataset_path):
    dataset_types = [
        "RGB",
        "RGB_NIR",
        "NDVI"
    ]
    # Create folders
    for dtype in dataset_types:
        os.makedirs(
            os.path.join(
                final_dataset_path,
                dtype
            ),
            exist_ok=True
        )
    # Process classes
    for category in sorted(os.listdir(data_path)):        
        category_path = os.path.join(
            data_path,
            category
        )
        if not os.path.isdir(category_path):
            continue
        print(f"\nProcessing {category}")
        for file in os.listdir(category_path):
            if not file.lower().endswith(".tif"):          
                continue
            file_path = os.path.join(category_path, file)
            base_name = os.path.splitext(file)[0]
            try:
                # RGB
                rgb = process_rgb_image(file_path).astype(np.float32)
                # RGB + NIR
                rgb_nir = create_rgb_nir(file_path).astype(np.float32)
                # NDVI
                ndvi = compute_ndvi(
                    rgb_nir,
                    nir_band=3,
                    red_band=0
                ).astype(np.float32)

                # Augmentation
                rgb_aug = augmentation(image=(rgb * 255).astype(np.uint8))["image"]

                rgb_aug = (rgb_aug.astype(np.float32)/ 255.0).clip(0, 1)

                # Create category folders
                for dtype in dataset_types:
                    os.makedirs(
                        os.path.join(
                            final_dataset_path,
                            dtype,
                            category
                        ),
                        exist_ok=True
                    )

                # Save RGB
                np.save(
                    os.path.join(
                        final_dataset_path,
                        "RGB",
                        category,
                        f"{base_name}.npy"
                    ),
                    rgb
                )

                np.save(
                    os.path.join(
                        final_dataset_path,
                        "RGB",
                        category,
                        f"{base_name}_aug.npy"
                    ),
                    rgb_aug
                )

                # Save RGB+NIR
                np.save(
                    os.path.join(
                        final_dataset_path,
                        "RGB_NIR",
                        category,
                        f"{base_name}.npy"
                    ),
                    rgb_nir
                )

                # Save NDVI
                np.save(
                    os.path.join(
                        final_dataset_path,
                        "NDVI",
                        category,
                        f"{base_name}.npy"
                    ),
                    ndvi
                )

            except Exception as e:
                print( f"Error processing {file}: {e}" )

# 6. Run Processing

In [9]:
save_all_processed_images(
    data_path=Data_path,
    final_dataset_path=final_dataset_path
)

print("\nProcessed dataset saved successfully.")
print(
    f"Location: {final_dataset_path}"
)


Processing AnnualCrop

Processing Forest

Processing HerbaceousVegetation

Processing Highway

Processing Industrial

Processing Pasture

Processing PermanentCrop

Processing Residential

Processing River

Processing SeaLake

Processed dataset saved successfully.
Location: D:\master\02- Second Semester\04- AI\Project\EuroSAT_MS\Processed_Data
